# hidden_profile

> Phase 13 (Track A, A3): the hidden-profile vote. `profileTask` deals facts so that the shared facts favour B and the private ones A; `ProfileEnv` runs rounds of posts on a shared board (facts published verbatim, or opinions) and a private vote; `soloCeiling` is the one-agent prompt with every fact. Imports only numpy (and the domains' data).

In [ ]:
#| default_exp hidden_profile

In [ ]:
#| hide
from nbdev.showdoc import *

A task deals the facts of one domain (option names, hidden-best option and fact order drawn from the seed). With the defaults (`shared=(1, 1, 3)`, `unique=2`, N = 4) the full totals are A +8 and B +3, the shared facts alone A 0 and B +3, and each agent alone A +2 and B +3: every agent prefers B until the private facts are pooled. `lone` gives one agent a fact that rules B out (sign -1, `ruleOut`).

In [ ]:
#| export
import numpy as np

from explRL.profile_domains import DOMAINS

_all_ = ['DOMAINS']


def profileTask(seed, variant='classic', domain=None, N=4, shared=(1, 1, 3), unique=2):
    """One hidden-profile task: names (shown order), best (the hidden-best option's index, A), facts (option, sign,
    text, holders, private, ruleOut), held[i] (agent i's fact indices in its own order, ids F1, F2, ...)."""
    assert variant in ('classic', 'lone'), variant
    rng = np.random.default_rng(seed)
    domain = domain or str(rng.choice(sorted(DOMAINS)))
    D = DOMAINS[domain]
    names = [str(x) for x in rng.choice(D['names'], 2, replace=False)]
    best = int(rng.integers(2))
    A, B = best, 1 - best
    aPos, aNeg, bPos = shared
    assert aPos + bPos + N * unique <= len(D['pos']) and aNeg <= len(D['neg']), 'not enough templates'
    pos, neg = iter(rng.permutation(D['pos'])), iter(rng.permutation(D['neg']))
    everyone = list(range(N))
    fact = lambda o, s, tpl, holders: dict(option=o, sign=s, text=str(tpl).format(o=names[o]), holders=holders,
                                           private=len(holders) < N, ruleOut=False)
    facts = [fact(A, 1, next(pos), everyone) for _ in range(aPos)] + [fact(A, -1, next(neg), everyone) for _ in range(aNeg)]
    facts += [fact(B, 1, next(pos), everyone) for _ in range(bPos)]
    facts += [fact(A, 1, next(pos), [i]) for i in range(N) for _ in range(unique)]
    if variant == 'lone':
        facts.append(dict(fact(B, -1, D['ruleOut'], [int(rng.integers(N))]), ruleOut=True))
    facts = [facts[k] for k in rng.permutation(len(facts))]
    held = [[k for k, f in enumerate(facts) if i in f['holders']] for i in range(N)]
    held = [[h[k] for k in rng.permutation(len(h))] for h in held]
    return dict(seed=seed, variant=variant, domain=domain, N=N, names=names, best=best, facts=facts, held=held,
                noun=D['noun'], question=D['question'])


def factTotals(task, idx=None):
    "Signed totals [option 0, option 1] over the facts `idx` (default: all)."
    t = [0, 0]
    for k in range(len(task['facts'])) if idx is None else idx:
        t[task['facts'][k]['option']] += task['facts'][k]['sign']
    return t

In [ ]:
#| export
_STR = {'type': 'string'}
POST_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['kind', 'fact', 'text', 'notes'],
               'properties': {'kind': {'type': 'string', 'enum': ['fact', 'opinion']}, 'fact': _STR, 'text': _STR,
                              'notes': _STR}}
DISSENT = 'If you hold information the others seem to lack, say so before the vote.'


def voteSchema(names):
    "The vote's schema: one of the two option names, and notes."
    return {'type': 'object', 'additionalProperties': False, 'required': ['vote', 'notes'],
            'properties': {'vote': {'type': 'string', 'enum': list(names)}, 'notes': _STR}}


def parseVote(names, parsed):
    "The index of the voted name (case and surrounding spaces ignored), or None."
    v = parsed.get('vote') if isinstance(parsed, dict) else None
    m = [k for k, n in enumerate(names) if isinstance(v, str) and v.strip().casefold() == n.casefold()]
    return m[0] if m else None


class ProfileEnv:
    "Rounds of posts on a shared board, then a private vote (the hidden-profile task of `profileTask`)."
    def __init__(self, task, rounds=3, rule='free', table=False, hint=None):
        assert rule in ('free', 'factsFirst') and hint in (None, 'dissent'), (rule, hint)
        self.task, self.rounds, self.rule, self.table, self.hint = task, int(rounds), rule, bool(table), hint
        self.SCHEMA, self.VOTE_SCHEMA = POST_SCHEMA, voteSchema(task['names'])
        self.round, self.board, self.pending, self.invalid = 1, [], {}, 0

    def rules(self):
        t, (x, y) = self.task, self.task['names']
        s = (f"You are one of {t['N']} members of a committee that decides {t['question']}: {x} or {y}. Each member "
             f"holds some facts about the two options; members may hold different facts. In each of {self.rounds} "
             "rounds every member posts once to a shared board: either one of their facts (kind 'fact', fact = its id, "
             "for example 'F2'; the board shows that fact's exact sentence) or an opinion (kind 'opinion', text = up to "
             "300 characters). Posts appear on the board after the round. After the last round every member votes "
             f"privately for {x} or {y}; the option with a majority of the votes is chosen.")
        if self.rule == 'factsFirst':
            s += ' In round 1 only facts may be posted: an opinion in round 1 is rejected.'
        if self.table:
            s += ' The board lists each fact posted so far once, grouped by option, without who posted it; opinions are listed below.'
        if self.hint == 'dissent':
            s += ' ' + DISSENT
        return s + (" Reply with one JSON object: kind, fact (the fact's id, or '' for an opinion), text (the opinion, "
                    "or '' for a fact) and notes (private notes to yourself, shown to you next round).")

    def ownFacts(self, i):
        return 'Your facts:\n' + '\n'.join(f"F{j + 1}: {self.task['facts'][k]['text']}" for j, k in enumerate(self.task['held'][i]))

    def boardText(self):
        if not self.table:
            return 'Board:\n' + ('\n'.join(f"round {p['round']}, agent {p['agent']} ({p['kind']}): {p['text']}"
                                           for p in self.board) or 'nothing posted yet.')
        posted, facts = sorted({p['fact'] for p in self.board if p['kind'] == 'fact'}), self.task['facts']
        lines = ['Facts posted so far, by option:']
        for o, n in enumerate(self.task['names']):
            fs = [facts[k]['text'] for k in posted if facts[k]['option'] == o]
            lines.append(f'About {n}:' + ''.join('\n- ' + x for x in fs) if fs else f'About {n}: none.')
        ops = [f"round {p['round']}, agent {p['agent']}: {p['text']}" for p in self.board if p['kind'] == 'opinion']
        return '\n'.join(lines + ['Opinions:'] + (ops or ['none.']))

    def observe(self, i, r=None):
        "Agent i's prompt in round r (default: the current round): its facts with ids, then the board (or the table)."
        r = self.round if r is None else r
        return f'Round {r} of {self.rounds}. You are agent {i}.\n{self.ownFacts(i)}\n{self.boardText()}'

    def post(self, i, parsed):
        "Queue agent i's post for this round; False (nothing posted, counted invalid) if the reply is not a valid post."
        kind = parsed.get('kind') if isinstance(parsed, dict) else None
        p = None
        if kind == 'fact':
            f = parsed.get('fact')
            ids = {f'F{j + 1}': k for j, k in enumerate(self.task['held'][i])}
            k = ids.get(f.strip().upper()) if isinstance(f, str) else None
            if k is not None:
                p = dict(round=self.round, agent=i, kind='fact', fact=k, text=self.task['facts'][k]['text'])
        elif kind == 'opinion' and not (self.rule == 'factsFirst' and self.round == 1):
            x = parsed.get('text')
            if isinstance(x, str) and x.strip():
                p = dict(round=self.round, agent=i, kind='opinion', fact=None, text=' '.join(x.split())[:300])
        if p is None:
            self.invalid += 1
            return False
        self.pending[i] = p
        return True

    def endRound(self):
        "Apply the round's posts in agent order (no agent sees a post of the same round)."
        self.board += [self.pending[i] for i in sorted(self.pending)]
        self.pending, self.round = {}, self.round + 1

    def voteText(self, i):
        x, y = self.task['names']
        return (f'The discussion is over. You are agent {i}.\n{self.ownFacts(i)}\n{self.boardText()}\n'
                f'Vote privately for {x} or {y}: reply with one JSON object, vote and notes.')

    def vote(self, i, parsed):
        "The option index agent i votes for, or None (invalid)."
        v = parseVote(self.task['names'], parsed)
        self.invalid += v is None
        return v

    def success(self, votes):
        "Strictly more than half of the N votes for the hidden-best option."
        return sum(v == self.task['best'] for v in votes) > self.task['N'] / 2

    def stats(self):
        "Distinct facts posted, facts in the task, private facts posted per private fact held, opinions, invalid replies."
        facts = self.task['facts']
        posted = {p['fact'] for p in self.board if p['kind'] == 'fact'}
        priv = [k for k, f in enumerate(facts) if f['private']]
        return dict(factsPosted=len(posted), factsHeld=len(facts),
                    uniquePosted=sum(k in posted for k in priv) / len(priv) if priv else float('nan'),
                    opinions=sum(p['kind'] == 'opinion' for p in self.board), invalid=self.invalid)


def soloCeiling(task):
    "The one-agent prompt: every fact, then the vote (schema `voteSchema(task['names'])`)."
    x, y = task['names']
    facts = '\n'.join('- ' + f['text'] for f in task['facts'])
    return (f"You decide alone {task['question']}: {x} or {y}. These are all the facts known about the two options:\n"
            f'{facts}\nVote for {x} or {y}: reply with one JSON object, vote and notes.')

## Tests

In [ ]:
from explRL.hidden_profile import *


def known(t, env, i):
    "Agent i's facts and the facts on the board."
    return set(t['held'][i]) | {p['fact'] for p in env.board if p['kind'] == 'fact'}


def play(t, rounds, posts, **kw):
    """A scripted team: posts(t, i, r) -> parsed post; each agent votes for the option with the higher total over the
    facts it knows (ties: the option shown first)."""
    env = ProfileEnv(t, rounds, **kw)
    for r in range(rounds):
        for i in range(t['N']):
            env.post(i, posts(t, i, r))
        env.endRound()
    votes = []
    for i in range(t['N']):
        tot = factTotals(t, known(t, env, i))
        votes.append(env.vote(i, dict(vote=t['names'][int(tot[1] > tot[0])], notes='')))
    return env, votes


def privateFirst(t, i, r):
    "Post the agent's own facts, private ones first."
    order = sorted(range(len(t['held'][i])), key=lambda j: not t['facts'][t['held'][i][j]]['private'])
    return dict(kind='fact', fact=f'F{order[r % len(order)] + 1}', text='', notes='')


silent = lambda t, i, r: dict(kind='opinion', fact='', text='I think we should decide carefully.', notes='')

# 1 000 seeds: shared facts alone favour B, all facts favour A (A +8, B +3), each agent alone prefers B (A +2, B +3);
# lone adds one fact, held by one agent, that rules B out; names, the hidden-best's position and fact order vary
pairs, bests, doms, orders = set(), set(), set(), set()
for seed in range(1000):
    t = profileTask(seed)
    A, B, N = t['best'], 1 - t['best'], t['N']
    sh = factTotals(t, [k for k, f in enumerate(t['facts']) if len(f['holders']) == N])
    al = factTotals(t)
    assert (sh[A], sh[B]) == (0, 3) and (al[A], al[B]) == (8, 3), seed
    for i in range(N):
        own = factTotals(t, t['held'][i])
        assert (own[A], own[B]) == (2, 3) and len(t['held'][i]) == 7
    assert all(f['option'] == A and f['sign'] == 1 and len(f['holders']) == 1 for f in t['facts'] if f['private'])
    texts = [f['text'] for f in t['facts']]
    assert len(set(texts)) == len(texts) == 13 and all(t['names'][f['option']] in f['text'] for f in t['facts'])
    L = profileTask(seed, 'lone')
    out = [f for f in L['facts'] if f['ruleOut']]
    assert (L['names'], L['best'], L['domain']) == (t['names'], t['best'], t['domain']) and len(L['facts']) == 14
    assert len(out) == 1 and len(out[0]['holders']) == 1 and out[0]['option'] == B and t['names'][B] in out[0]['text']
    assert sorted(f['text'] for f in L['facts'] if not f['ruleOut']) == sorted(texts)
    al = factTotals(L)
    assert al[A] > al[B]
    pairs.add(tuple(t['names'])); bests.add(A); doms.add(t['domain'])
    orders.add(tuple(f['option'] == A for f in t['facts']))
assert len(pairs) > 100 and bests == {0, 1} and len(doms) == 3 and len(orders) > 100
assert profileTask(7) == profileTask(7) and profileTask(7, domain='hiring')['domain'] == 'hiring'

In [ ]:
# a scripted team that posts every fact (enough rounds, or private facts first in 3 rounds) and votes by its totals
# always succeeds; one that posts no fact never does (classic variant)
for seed in range(1000):
    t = profileTask(seed)
    env, votes = play(t, 7, lambda t, i, r: dict(kind='fact', fact=f'F{r + 1}', text='', notes=''))
    assert env.stats()['factsPosted'] == 13 and env.success(votes) and env.stats()['uniquePosted'] == 1
    env, votes = play(t, 3, privateFirst)
    assert env.success(votes) and env.stats()['uniquePosted'] == 1 and env.stats()['invalid'] == 0
    env, votes = play(t, 3, silent)
    assert not env.success(votes) and sum(v == t['best'] for v in votes) == 0 and env.stats()['opinions'] == 12
    env, votes = play(profileTask(seed, 'lone'), 8, lambda t, i, r: dict(kind='fact', fact=f'F{r % len(t["held"][i]) + 1}', text='', notes=''))
    assert env.success(votes)

# posts: a fact id publishes the fact's own sentence; bad ids, kinds, empty opinions and non-dicts are invalid and
# post nothing; opinions are cut to 300 characters; factsFirst rejects round-1 opinions only
t = profileTask(1)
env = ProfileEnv(t)
op = dict(kind='opinion', fact='', text='x' * 400, notes='')
assert env.post(0, dict(kind='fact', fact=' f2 ', text='made up', notes='')) and env.pending[0]['text'] == t['facts'][t['held'][0][1]]['text']
bad = [dict(kind='fact', fact='F8', text='', notes=''), dict(kind='fact', fact=2, text='', notes=''), dict(kind='vote'),
       dict(kind='opinion', fact='', text='  ', notes=''), None, 'F1', {}]
assert not any(env.post(1, b) for b in bad) and 1 not in env.pending and env.invalid == len(bad)
assert env.post(2, op) and env.pending[2]['text'] == 'x' * 300
assert 'nothing posted yet' in env.observe(3)            # no same-round visibility
env.endRound()
assert [p['agent'] for p in env.board] == [0, 2] and env.round == 2 and 'round 1, agent 0 (fact): ' in env.observe(3)
ff = ProfileEnv(t, rule='factsFirst')
assert not ff.post(0, op) and ff.post(1, dict(kind='fact', fact='F1', text='', notes='')) and ff.invalid == 1
ff.endRound()
assert ff.post(0, op) and [p['kind'] for p in ff.board] == ['fact']

In [ ]:
# the table lists each posted fact once, grouped by option, without senders; opinions below
t = profileTask(2)
A, B = t['best'], 1 - t['best']
k = next(k for k, f in enumerate(t['facts']) if len(f['holders']) == t['N'] and f['option'] == B)
for table in (False, True):
    env = ProfileEnv(t, table=table)
    for i in range(3):
        env.post(i, dict(kind='fact', fact=f"F{t['held'][i].index(k) + 1}", text='', notes=''))
    env.post(3, dict(kind='opinion', fact='', text='We should pick carefully.', notes=''))
    env.endRound()
    o = env.boardText()
    assert o.count(t['facts'][k]['text']) == (1 if table else 3) and ('agent 0' in o) != table
    if table:
        assert o.index(f"About {t['names'][B]}:") < o.index(t['facts'][k]['text']) and f"About {t['names'][A]}: none." in o
        assert o.endswith('Opinions:\nround 1, agent 3: We should pick carefully.')
    assert env.stats()['factsPosted'] == 1 and env.stats()['opinions'] == 1

# votes: the schema's enum is the two names; a name (case and spaces ignored) parses, anything else is invalid
env = ProfileEnv(t)
assert env.VOTE_SCHEMA['properties']['vote']['enum'] == t['names'] == voteSchema(t['names'])['properties']['vote']['enum']
assert env.SCHEMA == POST_SCHEMA and POST_SCHEMA['properties']['kind']['enum'] == ['fact', 'opinion']
assert env.vote(0, dict(vote=f"  {t['names'][1].upper()} ", notes='')) == 1 and env.vote(0, dict(vote=t['names'][0])) == 0
assert env.vote(1, dict(vote='Nobody')) is None and env.vote(1, None) is None and env.invalid == 2
assert env.success([A, A, A, B]) and not env.success([A, A, B, B]) and not env.success([A, A, None, None])
assert parseVote(t['names'], dict(vote=t['names'][1])) == 1

# rules: each option adds its sentence only when on; observations show own facts with ids, the round and the board
base = ProfileEnv(t).rules()
assert all(x in base for x in t['names']) and 'only facts may be posted' not in base and DISSENT not in base
assert 'only facts may be posted' in ProfileEnv(t, rule='factsFirst').rules() and DISSENT in ProfileEnv(t, hint='dissent').rules()
assert 'once, grouped by option' in ProfileEnv(t, table=True).rules() and 'once, grouped by option' not in base
o = ProfileEnv(t, rounds=4).observe(2, 1)
assert o.startswith('Round 1 of 4. You are agent 2.\nYour facts:\nF1: ') and o.count('\nF') == len(t['held'][2])
assert ProfileEnv(t).voteText(1).endswith(f"Vote privately for {t['names'][0]} or {t['names'][1]}: reply with one JSON object, vote and notes.")

# the solo ceiling's prompt holds every fact
L = profileTask(2, 'lone')
s = soloCeiling(L)
assert all(f['text'] in s for f in L['facts']) and s.count('\n- ') == 14

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()